# 02 - tiling and normalisation check

**CPU runtime.** Confirms the preprocessing before any GPU time is spent.

| setting | value | why |
|---|---|---|
| tile | 512 px, no overlap | DINOv3's own segmentation protocol: 1024 patch tokens = 512 px at patch 16 |
| patch | 16 px -> 32 x 32 grid per tile | every DINOv3 ViT |
| `per_image` | z-score over the **whole image**, then tile | the study's first setting |
| `caffe` | official mean 0.3047 / std 0.3219 | the study's "one fixed global scaling" |
| `per_tile` | z-score per tile | legacy only: what the first (2026-09-22) run used by mistake |

Padding and the NA zone are excluded from labels and metrics.

In [1]:
# --- cell: mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# <<<SARTRANSFER-SYNC-BEGIN>>>
# Empty sync cell: published notebooks carry no packed code. On your own machine,
# in the repository folder, run   python sync.py   and open the notebook again;
# this cell then holds the package source and unpacks it on the runtime.
raise RuntimeError("The sync cell is empty. On your own machine, in the repository folder, "
                   "run:  python sync.py  and reopen this notebook.")
# <<<SARTRANSFER-SYNC-END>>>

In [3]:
# --- cell: data (zip on Drive, unpacked to local disk) ---
from pathlib import Path
from sartransfer.data import inventory as inv

DRIVE = Path("/content/drive/MyDrive")
assert DRIVE.exists(), "mount Drive first (cell 1)"
PROJECT = DRIVE / "sar-transfer"
RAW, RES = PROJECT / "raw", PROJECT / "results"
RES.mkdir(parents=True, exist_ok=True)
CAFFE = Path("/content/caffe")      # local disk: thousands of small PNGs are slow over Drive

inv.unzip_caffe(inv.fetch_caffe_zip(RAW), CAFFE)   # 2.86 GB download only the first time
DATA = inv.find_data_root(CAFFE)
print("data root:", DATA)

already present: /content/drive/MyDrive/sar-transfer/raw/data_raw.zip  (2.86 GB)
/content/caffe is not empty -- skipping extraction
data root: /content/caffe/data_raw


In [4]:
# --- cell: tile counts (expect 17,076 total / 14,620 train+val) ---
from PIL import Image
from sartransfer.data.tiling import TileSpec, n_tiles
from sartransfer.features import split_of

SPEC = TileSpec(size=512, overlap=0.0)
df = inv.scan(DATA, subdirs=("sar_images",))
df["size"] = [Image.open(p).size for p in df["path"]]        # (w, h), header only
df["tiles"] = [n_tiles(h, w, SPEC) for w, h in df["size"]]
df["split"] = df["glacier"].map(split_of)
print(df.groupby("split")["tiles"].agg(["size", "sum"]).rename(columns={"size": "images"}))
print("total tiles:", int(df["tiles"].sum()))

       images    sum
split               
test      122   2456
train     438  12595
val       121   2025
total tiles: 17076


In [5]:
# --- cell: the three normalisations on one real image ---
import numpy as np
from sartransfer.data.tiling import tile_image, normalise, normalise_image

path = df.sort_values("tiles").iloc[-1]["path"]           # a large image, many tiles
img = np.asarray(Image.open(path))
tiles, valid, _ = tile_image(img, SPEC)
print(Path(path).name, img.shape, "->", len(tiles), "tiles\n")

for mode in ("per_image", "caffe", "per_tile"):
    if mode == "per_tile":
        xt = normalise(tiles, mode, valid)
    else:
        xt = tile_image(normalise_image(img, mode), SPEC, fill=0.0)[0]
    means = np.array([t[v].mean() for t, v in zip(xt, valid) if v.any()])
    print(f"{mode:<10} spread of tile means: {means.std():.3f}"
          f"   (per_tile ~0 = brightness differences between tiles removed)")

COL_2016-05-20_TDX_7_1_092.png (2107, 4476) -> 45 tiles

per_image  spread of tile means: 0.275   (per_tile ~0 = brightness differences between tiles removed)
caffe      spread of tile means: 0.209   (per_tile ~0 = brightness differences between tiles removed)
per_tile   spread of tile means: 0.000   (per_tile ~0 = brightness differences between tiles removed)


In [6]:
# --- cell: patch targets on the same image ---
from sartransfer.data.targets import tile_patch_targets, CLASS_NAMES
from sartransfer.features import zone_path_for

zones = np.asarray(Image.open(zone_path_for(path, DATA)))
ztiles, _, _ = tile_image(zones, SPEC, fill=0)
labels, pvalid = tile_patch_targets(ztiles, valid, patch=16)
print("patch grid per tile:", labels.shape[1:], "| valid patches:",
      f"{pvalid.mean():.1%}")
counts = np.bincount(labels[pvalid], minlength=3)
print({n: int(c) for n, c in zip(CLASS_NAMES, counts)})

patch grid per tile: (32, 32) | valid patches: 74.0%
{'stone': 13518, 'glacier': 17453, 'ocean': 3148}
